# "But wait... there's more"

## A More Visible Agent Loop

The Digital Twin contained an Agent Loop. But it was behind-the-scenes, running every time the user asked a message. Using its tools and then replying. It didn't feel very... loopy.

### Adding 2 more ingredients to make it more real

Let's make an Agent Loop with some familiar features borrowed from Claude Code:

1. A Terminal UI (TUI)
2. A Checklist tool to cause and track multiple tool calls


In [3]:
# Start with some imports - rich is a library for making formatted text output in the terminal

from rich.console import Console
from dotenv import load_dotenv
from openai import OpenAI
import json
load_dotenv(override=True)

True

In [4]:
def show(text):
    try:
        Console().print(text)
    except Exception:
        print(text)

In [5]:
import os
GEMINI_BASE_URL = "https://generativelanguage.googleapis.com/v1beta/openai/"
openai = OpenAI(
    base_url=GEMINI_BASE_URL,
    api_key=os.getenv("GOOGLE_API_KEY") or os.getenv("GEMINI_API_KEY")
)

In [6]:
# Some lists!

checklist = []
completed = []

In [7]:
def get_checklist_report() -> str:
    result = ""
    for index, item in enumerate(checklist):
        if completed[index]:
            result += f"Checklist #{index + 1}: [green][strike]{item}[/strike][/green]\n"
        else:
            result += f"Checklist #{index + 1}: {item}\n"
    show(result)
    return result

In [8]:
get_checklist_report()

''

In [9]:
def create_checklist(descriptions: list[str]) -> str:
    checklist.extend(descriptions)
    completed.extend([False] * len(descriptions))
    return get_checklist_report()

In [10]:
def mark_complete(index: int, completion_notes: str) -> str:
    if 1 <= index <= len(checklist):
        completed[index - 1] = True
    else:
        return "No checklist at this index."
    Console().print(completion_notes)
    return get_checklist_report()

In [11]:
checklist, completed = [], []

create_checklist(["Buy groceries", "Finish week 1", "Eat banana"])

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: Buy groceries\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [12]:
mark_complete(1, "bought")

bought

Checklist #1: Buy groceries
Checklist #2: Finish week 1
Checklist #3: Eat banana

'Checklist #1: [green][strike]Buy groceries[/strike][/green]\nChecklist #2: Finish week 1\nChecklist #3: Eat banana\n'

In [13]:
create_checklist_json = {
    "name": "create_checklist",
    "description": "Add new checklist from a list of descriptions and return the full list",
    "parameters": {
        "type": "object",
        "properties": {
            "descriptions": {
                'type': 'array',
                'items': {'type': 'string'},
                'title': 'Descriptions of checklist items'
                }
            },
        "required": ["descriptions"],
        "additionalProperties": False
    }
}

In [14]:
mark_complete_json = {
    "name": "mark_complete",
    "description": "Mark complete the checklist item at the given position (starting from 1) and return the full list",
    "parameters": {
        'properties': {
            'index': {
                'description': 'The 1-based index of the checklist item to mark as complete',
                'title': 'Index',
                'type': 'integer'
                },
            'completion_notes': {
                'description': 'Notes about how you completed the checklist item in rich console markup',
                'title': 'Completion Notes',
                'type': 'string'
                }
            },
        'required': ['index', 'completion_notes'],
        'type': 'object',
        'additionalProperties': False
    }
}

In [15]:
tools = [{"type": "function", "function": create_checklist_json},
        {"type": "function", "function": mark_complete_json}]

In [16]:
def handle_tool_calls(tool_calls):
    results = []
    for tool_call in tool_calls:
        tool_name = tool_call.function.name
        arguments = json.loads(tool_call.function.arguments)
        tool = globals().get(tool_name)
        result = tool(**arguments) if tool else {}
        results.append({"role": "tool","content": json.dumps(result),"tool_call_id": tool_call.id})
    return results

In [19]:
def loop(messages):
    response = openai.chat.completions.create(model="gemini-3.5-flash", messages=messages, tools=tools)
    while response.choices[0].finish_reason == "tool_calls":
        message = response.choices[0].message
        tool_calls = message.tool_calls
        results = handle_tool_calls(tool_calls)
        messages.append(message)
        messages.extend(results)
        response = openai.chat.completions.create(model="gemini-3.5-flash", messages=messages, tools=tools)
    show(response.choices[0].message.content)

In [20]:
system_message = """
You are given a problem to solve, by using your checklist tools to plan a list of steps, then carrying out each step in turn.
Now create a plan, set the checklist, carry out the steps, and reply with the solution.
If any quantity isn't provided in the question, then include a step to come up with a reasonable estimate.
Provide your solution in Rich console markup without code blocks.
Do not ask the user questions or clarification; respond only with the answer after using your tools.
"""
user_message = """"
A train leaves Boston at 2:00 pm traveling 60 mph.
Another train leaves New York at 3:00 pm traveling 80 mph toward Boston.
When do they meet?
"""
messages = [{"role": "system", "content": system_message}, {"role": "user", "content": user_message}]

In [21]:
checklist, completed = [], []
loop(messages)

Checklist #1: Estimate the distance between Boston and New York
Checklist #2: Calculate the distance Train A (from Boston) travels between 2:00 PM and 3:00 PM
Checklist #3: Determine the remaining distance and relative speed of the two trains starting at 3:00 PM
Checklist #4: Calculate the meeting time based on the remaining distance and relative speed
Checklist #5: Format and present the final solution using Rich console markup

I estimated the distance between Boston and New York to be 200 miles. This is a highly realistic estimate (the 
actual driving distance is about 215 miles, and the flight distance is about 190 miles) and it simplifies the math 
nicely.

Checklist #1: Estimate the distance between Boston and New York
Checklist #2: Calculate the distance Train A (from Boston) travels between 2:00 PM and 3:00 PM
Checklist #3: Determine the remaining distance and relative speed of the two trains starting at 3:00 PM
Checklist #4: Calculate the meeting time based on the remaining distance and relative speed
Checklist #5: Format and present the final solution using Rich console markup

Train A travels for 1 hour (from 2:00 PM to 3:00 PM) before Train B starts. At a speed of 60 mph, Train A covers 60
miles during this hour.

Checklist #1: Estimate the distance between Boston and New York
Checklist #2: Calculate the distance Train A (from Boston) travels between 2:00 PM and 3:00 PM
Checklist #3: Determine the remaining distance and relative speed of the two trains starting at 3:00 PM
Checklist #4: Calculate the meeting time based on the remaining distance and relative speed
Checklist #5: Format and present the final solution using Rich console markup

...

The time required for them to meet after 3:00 PM is 140 miles / 140 mph = 1 hour. Adding 1 hour to 3:00 PM gives a 
meeting time of 4:00 PM.

Checklist #1: Estimate the distance between Boston and New York
Checklist #2: Calculate the distance Train A (from Boston) travels between 2:00 PM and 3:00 PM
Checklist #3: Determine the remaining distance and relative speed of the two trains starting at 3:00 PM
Checklist #4: Calculate the meeting time based on the remaining distance and relative speed
Checklist #5: Format and present the final solution using Rich console markup

Formatted the final answer beautifully using Rich console markup (bolding, underlining, colors) without using code 
blocks. I included the step-by-step solution for both a standard estimated distance of 200 miles and the actual 
rail distance of 230 miles.

Checklist #1: Estimate the distance between Boston and New York
Checklist #2: Calculate the distance Train A (from Boston) travels between 2:00 PM and 3:00 PM
Checklist #3: Determine the remaining distance and relative speed of the two trains starting at 3:00 PM
Checklist #4: Calculate the meeting time based on the remaining distance and relative speed
Checklist #5: Format and present the final solution using Rich console markup

InternalServerError: Error code: 503 - [{'error': {'code': 503, 'message': 'This model is currently experiencing high demand. Spikes in demand are usually temporary. Please try again later.', 'status': 'UNAVAILABLE'}}]

<table style="margin: 0; text-align: left; width:100%">
    <tr>
        <td style="width: 150px; height: 150px; vertical-align: middle;">
            <img src="../assets/exercise.png" width="150" height="150" style="display: block;" />
        </td>
        <td>
            <h2 style="color:#ff7800;">Exercise</h2>
            <span style="color:#ff7800;">Now try to build an Agent Loop from scratch yourself!<br/>
            Create a new .ipynb and make one from first principles, referring back to this as needed.<br/>
            It's one of the few times that I recommend typing from scratch - it's a very satisfying result.
            </span>
        </td>
    </tr>
</table>